### **Sieć neuronowa MTL na reprezentacji fingerprint - zestaw 1 - Absorpcja**

Wykorzystana reprezentacja: **ECFP4**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. AMES Mutagenicity

Wyniki dla STL:

In [1]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 144.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pytdc: filename=pytdc-1.1.15-py3-none-any.whl size=191643 sha256=ba80c54df63328bb7dfde74ec0c62604eb86535672743ccbf7c70c7f055ab370
  Stored in directory: /root/.cache/pip/wheels/3a/51/0f/0b00fd8b8288143eec76ea0a57804cddd72aae207cc4cb4d65
Successfully built pytdc
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
data_folder = "/content/drive/MyDrive/mldd_data" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

#data_folder = "/content/drive/MyDrive/MLDD - ADMET/data_splits"

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [4]:
class ADMETDescriptorFeaturizer:
    def __init__(self, y_column, smiles_col='Drug'):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.feature_names = [
            'MW', 'LogP', 'HBD', 'HBA', 'TPSA',
            'RotatableBonds', 'AromaticRings', 'HeavyAtoms',
            'MolMR', 'FractionCSP3'
        ]
        self.scaler = StandardScaler()

    def __call__(self, df, fit_scaler=True):
        features = []
        labels = []

        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            y = row[self.y_column] if self.y_column in df.columns else np.nan
            mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None

            if mol:
                desc_vector = [
                    Descriptors.MolWt(mol),
                    Descriptors.MolLogP(mol),
                    Lipinski.NumHDonors(mol),
                    Lipinski.NumHAcceptors(mol),
                    Descriptors.TPSA(mol),
                    Lipinski.NumRotatableBonds(mol),
                    Lipinski.NumAromaticRings(mol),
                    Descriptors.HeavyAtomCount(mol),
                    Descriptors.MolMR(mol),
                    Lipinski.FractionCSP3(mol)
                ]
            else:
                # Wstawiamy wektor zer o długości 10, jeśli cząsteczka jest błędna
                desc_vector = [0.0] * len(self.feature_names)

            features.append(desc_vector)
            labels.append(y)

        features_array = np.array(features)

        if fit_scaler:
            normalized_features = self.scaler.fit_transform(features_array)
        else:
            normalized_features = self.scaler.transform(features_array)

        return normalized_features, np.array(labels).reshape(-1, 1)

In [5]:

# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))


class MTL_NN_Hybrid(nn.Module):
    def __init__(self, input_dim, reg_tasks, class_tasks, hidden_dim=512):
        """
        reg_tasks: lista nazw zadań regresyjnych (np. ['Lipophilicity', 'Solubility'])
        class_tasks: lista nazw zadań klasyfikacyjnych (np. ['BBB', 'Ames'])
        """
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.reg_tasks = reg_tasks
        self.class_tasks = class_tasks

        # Słowniki głowic dla każdego typu zadania
        #Pozwala zapisać wiele warstw Linear i nazwać je tak, jak nazywają się Twoje zadania.
        self.reg_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in reg_tasks
        })
        self.class_heads = nn.ModuleDict({
            task: nn.Linear(hidden_dim, 1) for task in class_tasks
        })

    def forward(self, x):
        shared_features = self.encoder(x) #wspólny enkoder
        results = {}

        # Procesowanie zadań regresyjnych
        for task in self.reg_tasks:
            results[task] = self.reg_heads[task](shared_features)

        # Procesowanie zadań klasyfikacyjnych
        for task in self.class_tasks:
            # results[task] = torch.sigmoid(self.class_heads[task](shared_features))
            results[task] = self.class_heads[task](shared_features)

        #print(results)
        return results #Zwracając słownik:{'Caco2_Wang': 1.2, 'Lipophilicity_AZ': 0.5} masz absolutną pewność, który wynik dotyczy którego badania.

In [6]:

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

  #=============================

In [7]:
def train_MTL_hybrid_wl_sum(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- 1. Normalizacja (osobny skaler dla każdego zadania regresyjnego) ---
    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        # Wyciągamy dane i maskujemy NaN, aby skaler zadziałał
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        # Testowe przesyłamy w oryginale (skalujemy tylko przy ewaluacji dla wygody)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        # Klasyfikacja nie wymaga skalowania
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    # --- 2. Model i Optymalizator ---
    # Bierzemy wymiar wejściowy z liczby kolumn X_train_np
    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')

    print("\n--- START TRENINGU (Multi-Task) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        # Sumujemy straty ze wszystkich zadań, ignorując NaN
        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                total_loss += loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    # --- 3. Ewaluacja ---
    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            # Odwracamy skalowanie
            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()


            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask]) # <-- Dodaj tę linię
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,  # <-- I tę linię
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [8]:
def train_MTL_hybrid_uniform_average(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')


    print("\n--- START TRENINGU (Multi-Task - Uniform Average) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)

        task_losses = []

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask]).mean()
                task_losses.append(loss)

        if task_losses:
            total_loss = torch.stack(task_losses).mean() # Uniform average
        else:
            total_loss = torch.tensor(0.0, device=device, requires_grad=True)

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [9]:
def train_MTL_hybrid_uncertainty_weighting(X_train_np, X_test_np, y_train_dict, y_test_dict, reg_tasks, class_tasks):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    scalers = {}
    y_train_tensors = {}
    y_test_tensors = {}

    for task in reg_tasks:
        scalers[task] = StandardScaler()
        train_vals = y_train_dict[task].reshape(-1, 1)
        mask = ~np.isnan(train_vals).flatten()

        y_train_scaled = np.full_like(train_vals, np.nan)
        if mask.any():
            y_train_scaled[mask] = scalers[task].fit_transform(train_vals[mask])

        y_train_tensors[task] = torch.FloatTensor(y_train_scaled).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    for task in class_tasks:
        y_train_tensors[task] = torch.FloatTensor(y_train_dict[task].reshape(-1, 1)).to(device)
        y_test_tensors[task] = torch.FloatTensor(y_test_dict[task].reshape(-1, 1)).to(device)

    X_train_t = torch.FloatTensor(X_train_np).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = MTL_NN_Hybrid(input_dim=X_train_np.shape[1], reg_tasks=reg_tasks, class_tasks=class_tasks).to(device) #zmienione

    # Learnable parameters for task uncertainty
    log_vars = nn.ParameterDict()
    for task in reg_tasks + class_tasks:
        log_vars[task] = nn.Parameter(torch.zeros(1, device=device)) # Initialize log_sigma_sq to 0


    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion_reg = nn.MSELoss(reduction='none') # 'none' pozwala nam ręcznie obsłużyć NaN
    criterion_cls = nn.BCEWithLogitsLoss(reduction='none')


    print("\n--- START TRENINGU (Multi-Task - Uncertainty Weighting) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        total_loss = 0

        for task in reg_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_reg(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(0.5 * precision * loss + 0.5 * log_vars[task])

        for task in class_tasks:
            target = y_train_tensors[task]
            mask = ~torch.isnan(target)
            if mask.any():
                loss = criterion_cls(outputs[task][mask], target[mask])
                precision = torch.exp(-log_vars[task])
                total_loss += torch.mean(precision * loss + 0.5 * log_vars[task])

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Total Loss: {total_loss.item():.4f}")
            # print(f"    Log_vars: {{ {', '.join([f'{t}: {log_vars[t].item():.2f}' for t in log_vars])} }}")

    print("\n--- EWALUACJA ---")
    model.eval()
    all_metrics = {"reg_tasks": {}, "class_tasks": {}}

    with torch.no_grad():
        preds_dict = model(X_test_t)

        for task in reg_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_scaled = preds_dict[task].cpu().numpy()
            p_unscaled = scalers[task].inverse_transform(p_scaled).flatten()

            mse = mean_squared_error(y_true[mask], p_unscaled[mask])
            mae = mean_absolute_error(y_true[mask], p_unscaled[mask])
            r2 = r2_score(y_true[mask], p_unscaled[mask])

            all_metrics["reg_tasks"][task] = {
               "rmse": np.sqrt(mse),
               "mae": mae,
               "r2": r2
             }

        for task in class_tasks:
            y_true = y_test_dict[task].flatten()
            mask = ~np.isnan(y_true)
            if not mask.any(): continue

            p_probs  = torch.sigmoid(preds_dict[task]).cpu().numpy().flatten()
            p_labels = (p_probs[mask] > 0.5).astype(int)
            try:
                auroc = roc_auc_score(y_true[mask], p_probs[mask])
            except ValueError:
                auroc = 0.5

            all_metrics["class_tasks"][task] = {
                "accuracy": accuracy_score(y_true[mask], p_labels),
                "f1":       f1_score(y_true[mask], p_labels),
                "auroc":    auroc,
            }

    return all_metrics

In [10]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [11]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name: # Check if func_name is provided and not None
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [12]:
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer, fit_scaler=True):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    # TUTAJ JEST ZMIANA: dodajemy przekazanie parametru fit_scaler
    X_features, _ = featurizer(df_temp, fit_scaler=fit_scaler)

    # Sprawdzenie czy X zawiera NaN
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [13]:
def prepare_mtl_data_embeddings(df_list, task_names, dataset_names):
    """
    df_list: lista DataFrame ze splitów (np. [train_caco2, train_lipo])
    task_names: lista nazw zadań w modelu (np. ['Caco2_Wang', 'Lipophilicity_AZ'])
    dataset_names: lista nazw zbiorów odpowiadająca nazwom plików z embeddingami (np. ['Caco2_Wang', 'Lipophilicity_AstraZeneca'])
    """
    # 1. Zebranie wszystkich unikalnych struktur SMILES z przekazanych splitów
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Wczytanie embeddingów ze wszystkich powiązanych plików CSV dla danej konfiguracji
    # Budujemy jeden zbiorczy słownik mapujący Drug (SMILES) -> wektor embeddingu
    drug_to_emb = {}
    for d_name in dataset_names:
        file_path = f"{data_folder}/{d_name}_MoLFormer_embeddings.csv"
        try:
            emb_df = pd.read_csv(file_path)
            feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]
            for _, row in emb_df.iterrows():
                drug_to_emb[str(row['Drug'])] = row[feature_cols].values.astype(np.float32)
        except Exception as e:
            print(f"Błąd podczas ładowania embeddingów dla {d_name}: {e}")

    # 3. Filtrowanie master_list tylko do cząsteczek, które posiadają embeddingi
    safe_master_list = [drug for drug in sorted(list(all_drugs)) if drug in drug_to_emb]
    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 4. Tworzenie macierzy X_features z embeddingów
    emb_dim = len(next(iter(drug_to_emb.values()))) if drug_to_emb else 768
    X_features = np.zeros((n_samples, emb_dim), dtype=np.float32)
    for i, drug in enumerate(safe_master_list):
        X_features[i] = drug_to_emb[drug]

    # 5. Mapowanie etykiet y (dokładnie tak samo jak w wersji z deskryptorami)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [14]:
# # --- KONFIGURACJA I URUCHOMIENIE ---

# reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
# class_tasks = ['HIA_Hou']
# filepath = "mtl_results_absorpcja_embeddings.txt"

# # Definiujemy pełną listę nazw dla 4 plików z embeddingami
# dataset_names = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'AqSolDB', 'HIA_Hou']

# # 1. Ładowanie danych
# train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
# train_lipo,  test_lipo  = load_split_pickle('Lipophilicity_AstraZeneca')
# train_sol,   test_sol   = load_split_pickle('Solubility_AqSolDB')
# train_hia,   test_hia   = load_split_pickle('HIA_Hou')

# # 2. Przygotowanie danych MTL za pomocą embeddingów
# X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_lipo, train_sol, train_hia], reg_tasks + class_tasks, dataset_names)
# X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2,  test_lipo,  test_sol,  test_hia],  reg_tasks + class_tasks, dataset_names)

# loss_schemes = [
#     (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
#     (train_MTL_hybrid_uniform_average, "Uniform Average"),
#     (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
# ]

# # Pętla ucząca (loss_schemes) pozostaje całkowicie BEZ ZMIAN
# for train_func, scheme_name in loss_schemes:
#     results = train_func(
#         X_train_mtl,
#         X_test_mtl,
#         y_train_dict,
#         y_test_dict,
#         reg_tasks,
#         class_tasks=class_tasks
#     )

#     print("\n" + "="*40)
#     print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
#     print("="*40)

#     endpoint_group = str(reg_tasks + class_tasks)

#     for task_name, task_metrics in results['reg_tasks'].items():
#         formatted_wrapper = {"test_metrics": task_metrics}
#         print(f"\nWyniki dla zadania (Regresja): {task_name}")
#         print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
#         save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

#     for task_name, task_metrics in results['class_tasks'].items():
#         formatted_wrapper = {"test_metrics": task_metrics}
#         print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
#         print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
#         save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

# print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")

# Test1: Caco-2 (Wang) + Lipophilicity (Astra Zeneca)

Test pierwszy sprawdzający czy Lipophilicity (42000 związków) pomoże związanemu z nim CaCo-2 (Wang). Sprawdzamy czy MTL da lepsze wyniki niż STL.

In [15]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'Lipophilicity_AstraZeneca']

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_lipo], reg_tasks, dataset_names)
X_test_mtl, y_test_dict = prepare_mtl_data_embeddings([test_caco2, test_lipo], reg_tasks, dataset_names)

loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 2.4313
  Epoka  20 | Total Loss: 1.3301
  Epoka  40 | Total Loss: 0.9587
  Epoka  60 | Total Loss: 0.7241
  Epoka  80 | Total Loss: 0.5597

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4742
  MAE      : 0.3745
  R²       : 0.6460


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7710
  MAE      : 0.6126
  R²       : 0.5977


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1243
  Epoka  20 | Total Loss: 0.6213
  Epoka  40 | Total Loss: 0.4670
  Epoka  60 | Total Loss: 0.3514
  Epoka  80 | Total Loss: 0.2636

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Uniform Average
  RMSE     : 0.4811
  MAE      : 0.3774
  R²       : 0.6356


Wyniki dla

# Test2: Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik.

In [16]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'AqSolDB']

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo,  test_lipo  = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol,   test_sol   = load_split_pickle('Solubility_AqSolDB')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_lipo, train_sol], reg_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2,  test_lipo,  test_sol],  reg_tasks, dataset_names)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.4045
  Epoka  20 | Total Loss: 1.6031
  Epoka  40 | Total Loss: 1.2049
  Epoka  60 | Total Loss: 0.9445
  Epoka  80 | Total Loss: 0.7205

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4769
  MAE      : 0.3714
  R²       : 0.6420


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7334
  MAE      : 0.5718
  R²       : 0.6360


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.1840
  MAE      : 0.8722
  R²       : 0.7417


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.3233
  Epoka  20 | Total Loss: 0.5649
  Epoka  40 | Total Loss: 0.4059
  Epoka  60 | Total Loss: 0.3276
  Epoka  80 | Total Loss: 0.2578

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average



# Test3:  Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + HIA (Hou)

Sprawdzamy czy dołożenie kolejnego powiązanego zbioru poprawi wynik. Czy w jakimś momencie wyniki przestają się poprawiać?

In [17]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = ['HIA_Hou'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'AqSolDB', 'HIA_Hou']

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo,  test_lipo  = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol,   test_sol   = load_split_pickle('Solubility_AqSolDB')
train_hia,   test_hia   = load_split_pickle('HIA_Hou')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_lipo, train_sol, train_hia], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2,  test_lipo,  test_sol,  test_hia],  reg_tasks + class_tasks, dataset_names)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 4.6315
  Epoka  20 | Total Loss: 1.8425
  Epoka  40 | Total Loss: 1.2987
  Epoka  60 | Total Loss: 0.9404
  Epoka  80 | Total Loss: 0.6556

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4748
  MAE      : 0.3688
  R²       : 0.6451


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7338
  MAE      : 0.5655
  R²       : 0.6356


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.1501
  MAE      : 0.8505
  R²       : 0.7563


Wyniki dla zadania (Klasyfikacja): HIA_Hou

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.9741
  F1       : 0.9846
  AUROC    : 0.9957


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0824
  Epoka  20 | Total Loss: 0.4480
  Epoka  40 | Total Loss: 0

# Test4: Caco-2 (Wang) + HIA (Hou)

Sprawdzamy czy mały zbiór HIA pomoże CaCo-2 mniej niż większe zbiory z poprzednich testów. Jednocześnie sparwdzamy STL vs MTL.

In [18]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang']
class_tasks = ['HIA_Hou'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'HIA_Hou']

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_hia,   test_hia   = load_split_pickle('HIA_Hou')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_hia], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2,  test_hia],  reg_tasks + class_tasks, dataset_names)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.6645
  Epoka  20 | Total Loss: 0.6033
  Epoka  40 | Total Loss: 0.3752
  Epoka  60 | Total Loss: 0.1753
  Epoka  80 | Total Loss: 0.0768

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4748
  MAE      : 0.3732
  R²       : 0.6451


Wyniki dla zadania (Klasyfikacja): HIA_Hou

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.9569
  F1       : 0.9741
  AUROC    : 0.9913


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.5224
  Epoka  20 | Total Loss: 0.4245
  Epoka  40 | Total Loss: 0.2656
  Epoka  60 | Total Loss: 0.1785
  Epoka  80 | Total Loss: 0.1140

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Uniform Average
  RMSE     : 0.4885
  MAE      : 0.3822
  R²       : 0.6243


Wyniki dla zada

# Test5: Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)

Sprawdzamy czy duży zbiór Solubility pomoże innemu dużemu zbiorowi mniej niż małemu CaCo-2. Jednocześnie sprawdzamy STL vs MTL.

In [19]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = [] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Lipophilicity_AstraZeneca', 'AqSolDB']

# 1. Ładowanie danych
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol,  test_sol  = load_split_pickle('Solubility_AqSolDB')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_lipo, train_sol], reg_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_lipo,  test_sol],  reg_tasks, dataset_names)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.0525
  Epoka  20 | Total Loss: 1.4137
  Epoka  40 | Total Loss: 1.0318
  Epoka  60 | Total Loss: 0.8291
  Epoka  80 | Total Loss: 0.6819

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7627
  MAE      : 0.5986
  R²       : 0.6063


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.1840
  MAE      : 0.8867
  R²       : 0.7417


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1188
  Epoka  20 | Total Loss: 0.6150
  Epoka  40 | Total Loss: 0.5018
  Epoka  60 | Total Loss: 0.4088
  Epoka  80 | Total Loss: 0.3370

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Uniform Average
  RMSE     : 0.7699
  MAE      : 0.6090
  R²       : 0.598

# Test6: Caco-2 (Wang) + AMES Mutagenicity

Sprawdzamy czy zbiór niepowiązany pomoże zbiorowi CaCo-2.

In [20]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'AMES']

# 1. Ładowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_ames,  test_ames  = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_ames], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2,  test_ames],  reg_tasks + class_tasks, dataset_names)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 1.9684
  Epoka  20 | Total Loss: 1.0346
  Epoka  40 | Total Loss: 0.7777
  Epoka  60 | Total Loss: 0.6120
  Epoka  80 | Total Loss: 0.4774

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4826
  MAE      : 0.3799
  R²       : 0.6334


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7985
  F1       : 0.8154
  AUROC    : 0.8695


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.0212
  Epoka  20 | Total Loss: 0.5634
  Epoka  40 | Total Loss: 0.4501
  Epoka  60 | Total Loss: 0.3783
  Epoka  80 | Total Loss: 0.3282

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Uniform Average
  RMSE     : 0.4769
  MAE      : 0.3698
  R²       : 0.6419


Wyniki dla zadania

# Test7: Caco-2 (Wang)+ Lipophilicity (Astra Zeneca)  + AMES Mutagenicity

In [21]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ']
class_tasks = ['AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'AMES']

# 1. Ladowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_lipo, train_ames], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2, test_lipo, test_ames], reg_tasks + class_tasks, dataset_names)
loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 3.2860
  Epoka  20 | Total Loss: 1.9656
  Epoka  40 | Total Loss: 1.5245
  Epoka  60 | Total Loss: 1.2866
  Epoka  80 | Total Loss: 1.0975

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4639
  MAE      : 0.3668
  R²       : 0.6612


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.8021
  MAE      : 0.6387
  R²       : 0.5645


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7662
  F1       : 0.7818
  AUROC    : 0.8472


--- START TRENINGU (Multi-Task - Uniform Average) ---
  Epoka   0 | Total Loss: 1.1389
  Epoka  20 | Total Loss: 0.6345
  Epoka  40 | Total Loss: 0.4996
  Epoka  60 | Total Loss: 0.4130
  Epoka  80 | Total Loss: 0.3332

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Uniform Average

Wyniki dla

#Test 8: Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + HIA (Hou)  + AMES

In [22]:
# --- KONFIGURACJA I URUCHOMIENIE ---

reg_tasks = ['Caco2_Wang', 'Lipophilicity_AZ', 'Solubility_AqSolDB']
class_tasks = ['HIA_Hou', 'AMES'] # Dodaj tu zadania klasyfikacyjne
filepath = "mtl_results_absorpcja_embeddings.txt"

dataset_names = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'AqSolDB', 'HIA_Hou', 'AMES']

# 1. Ladowanie danych
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_hia, test_hia = load_split_pickle('HIA_Hou')
train_ames, test_ames = load_split_pickle('AMES')

# 2. Przygotowanie danych MTL
X_train_mtl, y_train_dict = prepare_mtl_data_embeddings([train_caco2, train_lipo, train_sol, train_hia, train_ames], reg_tasks + class_tasks, dataset_names)
X_test_mtl,  y_test_dict  = prepare_mtl_data_embeddings([test_caco2, test_lipo, test_sol, test_hia, test_ames], reg_tasks + class_tasks, dataset_names)

loss_schemes = [
    (train_MTL_hybrid_wl_sum, "Weighted Loss Sum"),
    (train_MTL_hybrid_uniform_average, "Uniform Average"),
    (train_MTL_hybrid_uncertainty_weighting, "Uncertainty Weighting")
]

# Loop through each loss weighting scheme
for train_func, scheme_name in loss_schemes:


    # 3. Uruchomienie treningu
    results = train_func(
        X_train_mtl,
        X_test_mtl,
        y_train_dict,
        y_test_dict,
        reg_tasks,
        class_tasks=class_tasks
    )

    # 4. RAPORTOWANIE I ZAPISYWANIE WYNIKOW

    print("\n" + "="*40)
    print(f"       PODSUMOWANIE MODELU MTL - {scheme_name}")
    print("="*40)

    endpoint_group = str(reg_tasks + class_tasks)

    # Przetwarzanie zadan regresyjnych
    for task_name, task_metrics in results['reg_tasks'].items():
        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Regresja): {task_name}")
        print_metrics(formatted_wrapper, task='regression', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='regression', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

    # Przetwarzanie zadan klasyfikacyjnych (jesli istnieja)
    for task_name, task_metrics in results['class_tasks'].items():

        # if 'f1' not in task_metrics:
        #     task_metrics['f1'] = 0.0  # Wypelnienie, jesli funkcja wymaga f1

        formatted_wrapper = {"test_metrics": task_metrics}

        print(f"\nWyniki dla zadania (Klasyfikacja): {task_name}")
        print_metrics(formatted_wrapper, task='classification', weight_loss_func_name=scheme_name)
        save_metrics(formatted_wrapper, task_name, filepath, task='classification', weight_loss_func_name=scheme_name, endpoint_group_name=endpoint_group)

print(f"\nWszystkie wyniki zostaly zapisane w: {filepath}")


--- START TRENINGU (Multi-Task) ---
  Epoka   0 | Total Loss: 5.0216
  Epoka  20 | Total Loss: 2.4367
  Epoka  40 | Total Loss: 1.9123
  Epoka  60 | Total Loss: 1.4744
  Epoka  80 | Total Loss: 1.1604

--- EWALUACJA ---

       PODSUMOWANIE MODELU MTL - Weighted Loss Sum

Wyniki dla zadania (Regresja): Caco2_Wang

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.4686
  MAE      : 0.3722
  R²       : 0.6543


Wyniki dla zadania (Regresja): Lipophilicity_AZ

  Loss Weighting: Weighted Loss Sum
  RMSE     : 0.7462
  MAE      : 0.5880
  R²       : 0.6232


Wyniki dla zadania (Regresja): Solubility_AqSolDB

  Loss Weighting: Weighted Loss Sum
  RMSE     : 1.1438
  MAE      : 0.8501
  R²       : 0.7589


Wyniki dla zadania (Klasyfikacja): HIA_Hou

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.9655
  F1       : 0.9794
  AUROC    : 0.9935


Wyniki dla zadania (Klasyfikacja): AMES

  Loss Weighting: Weighted Loss Sum
  Accuracy : 0.7909
  F1       : 0.8116
  AUROC    : 0.8637


--- STAR